In [1]:
import os, json, rich, unicodedata
from tqdm import tqdm
import spacy
from spacy.matcher import PhraseMatcher
from skillNer.general_params import SKILL_DB
from skillNer.skill_extractor_class import SkillExtractor
from skillNer.text_class import Text, Word
from itertools import islice


# nlp = spacy.load('spacy_outputs/model-best')
nlp = spacy.load('en_core_web_lg')
skill_extractor = SkillExtractor(nlp, SKILL_DB, PhraseMatcher)

loading full_matcher ...
loading abv_matcher ...
loading full_uni_matcher ...
loading low_form_matcher ...
loading token_matcher ...


#### skillNer
an opensource skill database and extractor; used to preannotate training data more reliably

In [2]:
with open('DATA/resumes.json','r') as f:
    data = json.loads(f.read())

In [3]:
def batch_docs(iterable, batch_size=50):
    it = iter(iterable) #yield sized chunks
    while True:
        batch = list(islice(it, batch_size))
        if not batch:
            break
        yield batch

In [ ]:
documents = []
i = 0

for batch in tqdm(batch_docs(data)):
    try:
        texts = [
            unicodedata.normalize('NFKD', d).encode('ascii', 'ignore').decode("utf-8") 
                    # ascii decode necessary to avoid index errors caused by unhandled spacy updates in skillNer
            for d in batch
        ]
        docs = list(nlp.pipe(texts))

        for text, doc in zip(texts, docs):
            entities = []
            doc_text = unicodedata.normalize('NFKD', text).encode('ascii', 'ignore').decode("utf-8")
            annotations = skill_extractor.annotate(doc_text)

            for n in annotations['results']['full_matches']:
                value = n['doc_node_value']
                label = SKILL_DB[n["skill_id"]]["skill_type"]
                start = doc_text.find(value)
                if start == -1:
                    continue
                else:
                    end = start + len(value)
                    entities.append([start, end, label, value])

            for m in annotations['results']['ngram_scored']:
                value = m['doc_node_value']
                start = m['doc_node_id'][0]
                end = start + len(m['doc_node_value'])
                label = SKILL_DB[m["skill_id"]]["skill_type"]
                entities.append([start, end, label, value])

            documents.append([doc_text, entities])

    except IndexError:
        with open(f'DATA/docs_{i}.json', 'x') as j:
            print('invalid document text')
            i += 1
            j.write(json.dumps(documents))

0it [00:00, ?it/s]/home/ronji/repos/CarP/ner-spacy-v2/ner-env/lib/python3.13/site-packages/skillNer/utils.py:99: UserWarning: [W008] Evaluating Token.similarity based on empty vectors.
  vec_similarity = token1.similarity(token2)
3it [37:43, 709.06s/it]

invalid document text


4it [46:28, 636.18s/it]

invalid document text


5it [51:29, 515.56s/it]

...

34it [15:55:13, 3183.01s/it]

invalid document text


36it [16:17:55, 1869.99s/it]

invalid document text


37it [16:35:12, 1620.17s/it]

In [9]:
print(len(documents))

11
